## THE CORRECT packages :/

In [2]:
!pip install --upgrade pip
!pip install open3d
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121
!pip install "numpy<2.0.0"

Looking in indexes: https://download.pytorch.org/whl/cu121


## Environment Setup

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

GOOGLE_DRIVE_PATH = "/content/drive/MyDrive/THESIS"
print(os.listdir(GOOGLE_DRIVE_PATH))

DATASET_PATH = "/content/drive/MyDrive/THESIS_other/dataset/rsu_1"

['open3d-ml', 'code', '.git', 'README.md']


## Import pretrained model to segment own pcd file
avoid loading semantickitti dataset

In [5]:
from posix import pipe
import open3d.ml as _ml3d
import open3d.ml.torch as ml3d
import torch
import open3d as o3d
import numpy as np

### input data preparation

In [66]:
sample_index = "017190"

pcd_path = os.path.join(DATASET_PATH, f"{sample_index}.pcd")
pcd = o3d.io.read_point_cloud(pcd_path)
points = np.asarray(pcd.points).astype(np.float32)
xyz = np.asarray(pcd.points)

# torch.tensor(np.float32(points))
# Set the points to the correct format for inference
data = {"point":torch.tensor(xyz),
        'feat': None,
        'label':torch.tensor(np.zeros((len(xyz),), dtype=np.int32))}

### Segmentation

In [7]:
framework = "torch"
device = "gpu" if torch.cuda.is_available() else "cpu"

# fetch config
# randlanet_semantickitti
cfg_file = os.path.join(GOOGLE_DRIVE_PATH, "open3d-ml/Open3D-ML/ml3d/configs/randlanet_semantickitti.yml")
cfg = _ml3d.utils.Config.load_from_file(cfg_file)

# model and pipeline
Model = _ml3d.utils.get_module("model", cfg.model.name, framework)
model = Model(**cfg.model)
pipeline = ml3d.pipelines.SemanticSegmentation(
    model,
    device=device,
    **cfg.pipeline
)

# weights
ckpt_folder = os.path.join(GOOGLE_DRIVE_PATH, "code", "logs")
os.makedirs(ckpt_folder, exist_ok=True)
ckpt_path = os.path.join(ckpt_folder, "randlanet_semantickitti_202201071330utc.pth")
randlanet_url = "https://storage.googleapis.com/open3d-releases/model-zoo/randlanet_semantickitti_202201071330utc.pth"
if not os.path.exists(ckpt_path):
    cmd = "wget {} -O {}".format(randlanet_url, ckpt_path)
    os.system(cmd)
pipeline.load_ckpt(ckpt_path=ckpt_path)

In [67]:
result = pipeline.run_inference(data)
print(result["predict_labels"])

test 0/1: 100%|██████████| 27515/27515 [02:45<00:00, 166.21it/s]


[ 4  4  4 ... 10 10 10]


/usr/local/lib/python3.12/dist-packages/open3d/_ml3d/torch/pipelines/semantic_segmentation.py:179: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(data['label']), model.cfg.num_classes,
/usr/local/lib/python3.12/dist-packages/open3d/_ml3d/torch/modules/metrics/semseg_metric.py:54: RuntimeWarning: Mean of empty slice
  accs.append(np.nanmean(accs))
/usr/local/lib/python3.12/dist-packages/open3d/_ml3d/torch/modules/metrics/semseg_metric.py:87: RuntimeWarning: Mean of empty slice
  ious.append(np.nanmean(ious))


### Add colors to visualize

In [9]:
COLOR_MAP = {
    -1: (192, 192, 192),  # light gray                  # unlabeled
    0: (255, 128, 0),     # bright green                # car
    1: (245, 150, 100),   # light orange / peach
    2: (245, 230, 100),   # yellow / light yellow
    3: (150, 60, 30),     # brown / dark brown
    4: (180, 30, 80),     # dark pink / reddish magenta
    5: (255, 0, 0),       # pure red
    6: (30, 30, 255),     # bright blue
    7: (200, 40, 255),    # purple / violet
    8: (90, 30, 150),     # dark purple                 # close ground
    9: (255, 0, 255),     # magenta / fuchsia           # front right ground
    10: (255, 150, 255),  # light pink / light magenta  # front right ground
    11: (75, 0, 75),      # very dark purple            # far ground
    12: (75, 0, 175),     # deep violet / dark blue-purple  # building
    13: (0, 200, 255),    # cyan / sky blue             # car and curbs
    14: (0, 175, 0),      # dark green                  # tree
    15: (50, 20, 150),    # indigo / deep purple-blue
    16: (0, 60, 135),     # dark blue                   # curbs
    17: (0, 128, 255),    # bright sky blue             # buildings
    18: (150, 240, 255),  # very light cyan / pale blue
    19: (0, 0, 255),      # pure blue
}


COLOR_MAP = {k: tuple(np.array(v)/255.0) for k, v in COLOR_MAP.items()}

In [68]:
points = xyz
labels_full = result["predict_labels"]
point_colors_full = np.array([COLOR_MAP[l] for l in labels_full])

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(point_colors_full)
o3d.io.write_point_cloud(f"/content/drive/My Drive/THESIS/code/example_result/segmentation_lidar_full_{sample_index}.ply", pcd)

True

In [69]:
# static objects (buildings & poles)
labels_object = -1 * np.ones(len(labels_full), dtype=int)
# labels_object[(labels_full == 0)] = 0  # car
labels_object[(labels_full == 17)] = 17  # building
labels_object[(labels_full == 12)] = 17  # building
labels_object[(labels_full == 16)] = 16  # curbs

In [70]:
# moving objects (cars)
road1 = {
    'xpymin': 0,
    'xpymax': 15,
    'ymxmin': -60,
    'ymxmax': 10
}
road2 = {
    'ymxmin': -12,
    'ymxmax': -2.3,
    'xpymin': -10,
    'xpymax': 60
}
zmin, zmax = -3.8, 30

for i in range(len(labels_object)):
    # road 1
    if road1['xpymin'] < xyz[i][0] + xyz[i][1] < road1['xpymax'] and road1['ymxmin'] < xyz[i][1] - xyz[i][0] < road1['ymxmax'] and zmin < xyz[i][2] < zmax:
        labels_object[i] = 0
    # road 2
    if road2['ymxmin'] < xyz[i][1] - xyz[i][0] < road2['ymxmax'] and road2['xpymin'] < xyz[i][0] + xyz[i][1] < road2['xpymax'] and zmin < xyz[i][2] < zmax:
        labels_object[i] = 0

In [71]:
unique_labels_car, counts = np.unique(labels_object, return_counts=True)

print("Labels:", unique_labels_car)
print("Counts:", counts)

point_colors = np.array([COLOR_MAP[l] for l in labels_object])

pcd_part = o3d.geometry.PointCloud()
pcd_part.points = o3d.utility.Vector3dVector(points)
pcd_part.colors = o3d.utility.Vector3dVector(point_colors)
o3d.io.write_point_cloud(f"/content/drive/My Drive/THESIS/code/example_result/segmentation_lidar_part_{sample_index}.ply", pcd_part)

Labels: [-1  0 16 17]
Counts: [22221  1676  1772  2314]


True